# Market + Résumé Power Ratings: methodology

This notebook documents the full model and reloads its live inputs and outputs, so the numbers here always match the latest run.

**Refresh:** `.venv/bin/python scripts/refresh_methodology.py` re-executes this notebook in place against the latest snapshot. Prose describes *how* the model works and only needs editing when the method changes. When it does, add a line to the changelog at the end.

**Contents**
1. What a rating means
2. Data
3. Stage A: one implied margin per game
4. The market rating
5. The résumé rating (in detail)
6. The blend
7. Ranks, ties and win % vs. average
8. Season simulation and playoff odds
9. Validation: what was tested, what held up
10. Known limitations
11. Changelog

In [1]:
import sys, yaml, numpy as np, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "docs" else Path.cwd()
sys.path.insert(0, str(ROOT))
pd.set_option("display.width", 160); pd.set_option("display.precision", 2)

SNAPS = ROOT / "data" / "snapshots"
STEM = sorted(p.name.replace("_stage_b.parquet", "") for p in SNAPS.glob("*_stage_b.parquet"))[-1]
B = pd.read_parquet(SNAPS / f"{STEM}_stage_b.parquet").set_index("team")
SIM = pd.read_parquet(SNAPS / f"{STEM}_sim.parquet").set_index("team")
SEASON = int(STEM[:4])
print(f"Latest rankings snapshot: {STEM} (season {SEASON}, résumé through week {int(B.blend_cutoff_week.iloc[0])})")

Latest rankings snapshot: 20261013T133003Z (season 2026, résumé through week 5)


## 1. What a rating means

A team's **rating** is its expected point margin against a league-average team on a neutral field. Ratings sum to zero across the league. **Win % vs. average** turns a rating into the chance of beating that average team, using the Stage A margin model (section 3).

The published rating blends two estimates of the same thing:

* **Market rating:** what prediction-market prices imply about each team's strength from here on.
* **Résumé rating:** what the team's results so far imply, adjusted for opponents.

The blend weights were measured on 27 seasons of history (section 6), not chosen by hand. No expert rankings or preseason projections enter directly; markets carry their own priors.

## 2. Data

| Source | Used for |
|---|---|
| **Kalshi** public API (`external-api.kalshi.com/trade-api/v2`), no key | Game moneylines (`KXNFLGAME`), spread ladders (`KXNFLSPREAD`), season win totals (`KXNFLWINS`), division / conference / Super Bowl / #1 seed futures, hourly price history |
| **nflverse** via `nflreadpy` | Schedules, results, closing spreads and totals (1999 on), play-by-play (EPA, win probability, fumbles, passes defended) |

Series tickers are discovered from Kalshi's `/series` listing and pinned in `config/series.yaml`. A daily job (`scripts/daily.py`, 9:30 a.m. ET) snapshots every open market. Prices are bid/ask **midpoints**, never last trades.

## 3. Stage A: one implied margin per game

Every Kalshi game market becomes one statement about the home margin *M*: "P(home margin beats *x*) = *q*". A push at *x* counts half, matching Kalshi's $0.50 settlement of tied moneylines.

* home spread rung "wins by over *t*" → *x* = *t*
* away rung → *x* = −*t*, *q* = 1 − price
* moneylines → *x* = 0

**Margin model.** Margins are integers. P(*M* = *m*) is a normal curve, bucketed to integers, reweighted by a **key-number weight** *w*(|*m*|) estimated from 2015–2025 results against closing spreads. That way 3, 7, 10 and 14 carry their real share and ties are rare. Beyond 21 the weights share one tail value.

Each game's location and σ are fitted to its ladder by weighted least squares, with weights 1/width² so tight quotes count more. Moneyline-only games use the pooled σ. Quotes at ≤ 2¢ or ≥ 98¢ are dropped. The output per game is the **expected home margin (mean)**, which is what the ratings use, and the **fair line (median)**, which is comparable to a sportsbook spread. On the first snapshot, the fair line matched the book within 0.5 points for all 16 ladder games.

In [2]:
w = pd.read_csv(ROOT / "data" / "key_number_weights.csv", comment="#")
print(open(ROOT / "data" / "key_number_weights.csv").readline().strip())
print("Key-number weights (1 = plain normal):")
print(w.set_index("abs_margin").weight.loc[["0", "1", "2", "3", "4", "6", "7", "10", "14", ">21"]].round(2).to_string())
sa = pd.read_parquet(SNAPS / f"{STEM}_stage_a.parquet")
print(f"\nThis snapshot: {len(sa)} Kalshi games, pooled sigma {sa.loc[sa.sigma_source=='pooled','sigma'].iloc[0] if (sa.sigma_source=='pooled').any() else sa.sigma.median():.2f}")
sa[["game_id", "mean", "fair_line", "sigma", "sigma_source", "rmse_cents", "book"]].head(6)

# seasons 2015-2025, 3028 games, fitted sigma 12.850
Key-number weights (1 = plain normal):
abs_margin
0      0.09
1      0.63
2      0.64
3      2.10
4      0.67
6      1.05
7      1.35
10     0.84
14     1.15
>21    0.81

This snapshot: 28 Kalshi games, pooled sigma 12.25


,game_id,mean,fair_line,sigma,sigma_source,rmse_cents,book
0,2026_06_SEA_DEN,-2.64,-2.84,12.91,ladder,0.94,-2.5
1,2026_06_ARI_LA,13.34,13.67,11.25,ladder,1.24,13.5
2,2026_06_BAL_CLE,-5.20,-4.40,12.43,ladder,0.98,-4.5
3,2026_06_BUF_LV,-6.15,-5.56,11.56,ladder,1.08,-6.0
4,2026_06_CAR_PHI,0.32,0.82,13.12,ladder,0.51,1.5
5,2026_06_CHI_ATL,-3.46,-3.15,12.66,ladder,0.71,-3.0


## 4. The market rating

Model: expected home margin = *r*<sub>home</sub> − *r*<sub>away</sub> + *H* (*H* = 0 at neutral sites), Σ*r* = 0.

Inputs, all fitted at once (`ratings/wins.py: fit_joint`):

1. **Current Kalshi game lines:** Stage A means for every listed game, typically the next 1–2 weeks.
2. **Past closing lines:** nflverse closing spreads for played games, converted from median to mean with the same margin model, and down-weighted with a **3-week half-life**. These keep the league connected early in the season; current lines alone split it into disconnected groups of teams.
3. **Season win totals (`KXNFLWINS`):** each team's final-wins distribution is *exact*: current wins plus a Poisson-binomial over its real remaining schedule. Each team's market ladder is compared with it as a distribution (KL divergence over the buckets the quoted rungs define), weighted by λ.

Lines enter as a Gaussian likelihood (2-point noise), win totals as λ × KL. **λ = 300** was chosen by holding out the current Kalshi lines and predicting them; win totals cut that error by about 40% (3.29 → 1.91 points). Home-field advantage is held at the lines-only estimate, because win totals barely identify it.

Futures (division, conference, Super Bowl, #1 seed) are **not** in the published market rating. A full joint fit through the season simulation exists (`scripts/joint_fit.py`) but is used as a diagnostic.

In [3]:
wt = yaml.safe_load(open(ROOT / "config" / "win_totals.yaml"))
print(f"lambda = {wt['lambda']:g}; held-out RMSE by lambda: {wt['heldout_rmse']}")
m = B[["record", "market", "E_wins", "E_wins_mkt"]].sort_values("market", ascending=False)
print(f"\nHFA {B.hfa.iloc[0]:.2f}; mean |model - market| expected wins: {(m.E_wins - m.E_wins_mkt).abs().mean():.2f}")
m.head(10)

lambda = 300; held-out RMSE by lambda: {'0': 3.2862, '1': 2.998, '3': 2.6885, '10': 2.2989, '30': 2.0571, '100': 1.9441, '300': 1.911, '1000': 1.8999, '3000': 1.8969}

HFA 1.76; mean |model - market| expected wins: 0.07


,record,market,E_wins,E_wins_mkt
team,,,,
LA,2-3,6.48,10.25,10.41
BUF,4-1,5.20,12.12,12.19
SEA,3-2,4.59,10.19,10.33
BAL,4-1,4.56,11.65,11.83
KC,4-0,4.07,11.69,11.82
JAX,4-1,4.02,11.51,11.59
SF,5-0,3.48,12.14,12.23
HOU,1-4,2.59,8.10,8.16
CHI,3-2,2.28,9.67,9.80


## 5. The résumé rating

The résumé answers: *given the games played so far, how good has this team been, adjusted for who it played?* It uses **final margins only**: no yardage, no EPA (section 9 explains why).

**Fit** (`ratings/backtest.py: resume_ratings`): weighted ridge least squares over all games through the last completed week:

margin<sub>g</sub> = *s*<sub>home</sub> − *s*<sub>away</sub> + *H*<sub>résumé</sub> + error

* **Blowout cap:** margins are clipped at ±21, so running up the score isn't rewarded.
* **Recency:** each game's weight halves every 3 weeks (weight = 0.5<sup>age / 3 weeks</sup>, age measured from the next week's kickoff).
* **Opponent adjustment:** comes from the joint fit itself. Beating a team whose own results are strong counts more. Opponent strength here is results-based, not market-based, so the résumé stays a pure results measure.
* A tiny ridge penalty (10⁻³) keeps week-1 fits solvable.

**Luck-neutral résumé.** The same fit on *margin − luck points*, where luck points measure turnover luck in the home team's favour:

* **Fumble luck:** recoverable fumbles in the game (out-of-bounds excluded) are coin flips; offenses kept 50.2% of their own fumbles in 2022–2025. Each recovery above 50% is worth **4.16 points**, the expected-points (EPA) gap between keeping and losing a fumble, from 2,021 fumbles.
* **Interception luck:** every interception is also recorded as a pass defended. A defense "should" intercept a share *s* of its passes defended, where *s* is the trailing 3-season league rate (about 22% around 2000, 17% in 2025; it drifts, so it isn't fixed). Each interception above *s* × passes defended is worth **3.40 points** (EPA of an interception vs. a broken-up incompletion).

**Close-game record and turnover margin** are reported in the tables for context, but are not part of any rating.

In [4]:
from ratings.features import FUMBLE_SWING_PTS, INT_SWING_PTS, season_features
from ratings.pbp_cache import load_pbp_cached
from ratings.schedule import load_schedule
from ratings.linear import decay_weight
print(f"fumble swing {FUMBLE_SWING_PTS} pts, interception swing {INT_SWING_PTS} pts")
feats = season_features(range(SEASON - 3, SEASON + 1), load_pbp_cached)
k = int(B.blend_cutoff_week.iloc[0])
f = feats[(feats.season == SEASON) & (feats.week <= k)].copy()
f["weight"] = decay_weight((k + 1 - f.week) * 7, 3.0)
TEAM = B.resume.idxmax()
g = f[(f.home_team == TEAM) | (f.away_team == TEAM)].copy()
sign = np.where(g.home_team == TEAM, 1, -1)
g = pd.DataFrame({"week": g.week, "opponent": np.where(sign > 0, g.away_team, g.home_team),
                  "margin": sign * g.margin, "capped": np.clip(sign * g.margin, -21, 21),
                  "luck_pts": sign * g.luck_pts, "luck_neutral": np.clip(sign * g.margin_neutral, -21, 21),
                  "weight": g.weight}).sort_values("week")
print(f"Example: {TEAM}, the top résumé (rating {B.resume[TEAM]:+.2f}; luck-neutral {B.resume_neutral[TEAM]:+.2f})")
g

fumble swing 4.16 pts, interception swing 3.4 pts


Example: JAX, the top résumé (rating +8.85; luck-neutral +8.62)


,week,opponent,margin,capped,luck_pts,luck_neutral,weight
821,1,CLE,24,21,4.88,19.12,0.31
838,2,DEN,-7,-7,2.42,-9.42,0.40
859,3,NE,29,21,4.00,21.00,0.50
871,4,CIN,5,5,3.78,1.22,0.63
892,5,PHI,2,2,-2.80,4.80,0.79


In [5]:
r = B[["record", "resume", "resume_neutral", "close", "fum_pts", "to_margin"]].copy()
r["resume_rank"] = r.resume.rank(ascending=False).astype(int)
r["neutral_rank"] = r.resume_neutral.rank(ascending=False).astype(int)
r.sort_values("resume", ascending=False)

,record,resume,resume_neutral,close,fum_pts,to_margin,resume_rank,neutral_rank
team,,,,,,,,
JAX,4-1,8.85,8.62,2-1,2.08,5,1,3
NE,3-2,8.53,9.80,1-1,-2.08,-5,2,1
KC,4-0,7.34,3.58,2-0,10.40,5,3,12
MIN,5-0,7.14,5.09,3-0,4.16,6,4,5
SF,5-0,6.64,4.77,2-0,-6.24,2,5,7
CAR,2-2,4.72,4.30,1-1,6.24,3,6,11
CIN,3-2,4.63,4.58,2-2,0.00,2,7,9
GB,3-2,4.40,1.33,2-0,4.16,3,8,14
ATL,2-3,4.24,5.77,0-2,4.16,-5,9,4


## 6. The blend

**rating = *b*<sub>market</sub>(*k*) × market + *b*<sub>résumé</sub>(*k*) × résumé + *b*<sub>neutral</sub>(*k*) × luck-neutral résumé**, centred to sum to zero. *k* is the last completed week.

**Calibration** (`scripts/calibrate_blend.py`). For every season 1999–2025 and every cutoff week 2–12, build the market rating (closing lines through week *k*+1, 3-week half-life) and both résumés (through week *k*). Then regress every later game's margin on the three predictions. Each weight is a **quadratic curve in *k***, fitted jointly across weeks, so neighbouring weeks share data and the ranking doesn't jump when the method changes.

Reading the weights:
* **b<sub>market</sub> < 1** is regression to the mean over the rest of the season. It scales every team equally and doesn't change the order.
* **Total résumé weight** (*b*<sub>résumé</sub> + *b*<sub>neutral</sub>) grows from about 0 at week 2 to about 0.38 by week 12.
* **Luck is treated as real early and discounted later:** the luck-neutral term takes over from about week 7 to week 10. In 3–4 game samples, what looks like luck partly isn't.

Out of sample (leave-one-season-out), this curve matched or beat separate per-week fits and a hard switch at week 6.

Caveat: the curve was calibrated against a lines-only market rating (history has no win-total prices). Today's market rating also uses win totals and is more accurate, so the résumé weights may be slightly generous.

In [6]:
from ratings.blend import load_blend
cfg = yaml.safe_load(open(ROOT / "config" / "resume_blend.yaml"))
wk = pd.DataFrame(cfg["by_cutoff"]).set_index("cutoff_week")[["a", "b_market", "b_margin", "b_neutral"]]
wk["résumé total"] = wk.b_margin + wk.b_neutral
bl = load_blend(k)
print(f"This week (cutoff {k}): {bl.b_market:.3f} x market + {bl.b_margin:.3f} x résumé + {bl.b_neutral:.3f} x luck-neutral")
wk.apply(lambda col: col.map(lambda v: f"{v:.3f}"))

This week (cutoff 5): 0.752 x market + 0.138 x résumé + 0.057 x luck-neutral


,a,b_market,b_margin,b_neutral,résumé total
cutoff_week,,,,,
2,0.352,0.860,0.062,-0.047,0.016
3,0.306,0.819,0.101,-0.019,0.082
4,0.279,0.783,0.126,0.016,0.142
5,0.269,0.752,0.138,0.057,0.195
6,0.277,0.727,0.137,0.104,0.241
7,0.302,0.706,0.122,0.158,0.280
8,0.345,0.690,0.095,0.218,0.313
9,0.406,0.680,0.055,0.284,0.339
10,0.484,0.674,0.001,0.357,0.358


In [7]:
d = B[["record", "market", "resume", "resume_neutral", "blend"]].copy()
d["market part"] = bl.b_market * d.market
d["résumé part"] = bl.b_margin * d.resume + bl.b_neutral * d.resume_neutral
d["market part"] -= d["market part"].mean(); d["résumé part"] -= d["résumé part"].mean()
print("Top 10, rating = market part + résumé part (centred):")
d.sort_values("blend", ascending=False).head(10)[["record", "market part", "résumé part", "blend"]]

Top 10, rating = market part + résumé part (centred):


,record,market part,résumé part,blend
team,,,,
LA,2-3,4.88,0.16,5.04
BUF,4-1,3.91,1.09,4.99
JAX,4-1,3.02,1.71,4.73
KC,4-0,3.06,1.21,4.28
BAL,4-1,3.43,0.51,3.95
SF,5-0,2.62,1.19,3.81
SEA,3-2,3.46,-0.09,3.37
NE,3-2,1.39,1.73,3.12
CHI,3-2,1.71,0.74,2.46


## 7. Ranks, ties and win % vs. average

* Teams whose ratings **round to the same displayed value** (one decimal) share a rank, shown as "T-1, T-1, 3, …" (`ratings/display.py`).
* Ratings carry roughly ±1 point of uncertainty, so neighbouring ranks within a point or so are effectively ties even when the displayed values differ.
* Win % vs. average uses the Stage A margin model at the pooled σ. Expected margin converts to win probability the same way for every team. This was checked directly: on 7,000 games, a spread converts to the same win rate in low-total and high-total games, so defense-first teams aren't shortchanged.

In [8]:
from ratings.display import display_ranks
B.assign(shown=display_ranks(B.blend)).sort_values("blend", ascending=False)[["shown", "record", "blend"]].head(8)

,shown,record,blend
team,,,
LA,T-1,2-3,5.04
BUF,T-1,4-1,4.99
JAX,3,4-1,4.73
KC,4,4-0,4.28
BAL,5,4-1,3.95
SF,6,5-0,3.81
SEA,7,3-2,3.37
NE,8,3-2,3.12


## 8. Season simulation and playoff odds

`ratings/season_sim.py` simulates the remaining schedule 10,000 times:

* Each game is drawn as a home win, away win or tie, with probabilities from the Stage A margin model at the game's expected margin. Ties occur at their real rate of about 0.3%.
* **Rating uncertainty τ:** each team gets one N(0, τ) shift per simulated season. τ is the futures-blind fit's estimate from win totals (about 4.7–5.0 points). It also fits the Kalshi futures best, independently.
* **Tiebreakers:**
  * Division: head-to-head, division record, common games, conference record, strength of victory, strength of schedule, coin flip.
  * Wild cards: one club per division, head-to-head sweep, conference record, common games (minimum 4), strength of victory, strength of schedule, coin flip.
  * When a step separates a 3+ team tie, the rest restart at step 1.
* 7 seeds per conference. Higher seed hosts; #1 plays the lowest remaining seed.

**Validated** on actual 2020–2025 results (`scripts/validate_tiebreakers.py`): it reproduces every playoff field, division winner, #1 seed and wild-card pairing, 12 of 12 conference-seasons.

In [9]:
SIM.sort_values("sb", ascending=False)[["mean_wins", "playoffs", "division", "seed_1", "conf", "sb"]].head(10)

,mean_wins,playoffs,division,seed_1,conf,sb
team,,,,,,
BUF,11.88,0.88,0.65,0.24,0.18,0.10
LA,9.98,0.66,0.21,0.08,0.16,0.09
SF,12.06,0.87,0.59,0.35,0.18,0.09
BAL,11.45,0.80,0.56,0.18,0.16,0.09
KC,11.59,0.85,0.64,0.22,0.16,0.09
JAX,11.37,0.83,0.70,0.15,0.14,0.07
SEA,10.07,0.62,0.19,0.08,0.12,0.06
NE,9.97,0.68,0.31,0.06,0.08,0.04
DET,9.98,0.61,0.27,0.09,0.08,0.04


## 9. Validation: what was tested, what held up

**Résumé features (1999–2025, leave-one-season-out).** Added to the market rating, final margin was the best single résumé input through week ~5, and luck-neutral margin was best from week ~6. Net EPA/play, success rate, garbage-time-filtered EPA, average win probability and post-game win expectancy ("deserved margin") never beat margin once margin was in. Combining all features overfit. The best recency half-life was about 3 weeks.

**Market efficiency.**
* Against each game's closing spread, neither the résumé, recent form against the spread, nor "playing down to the competition" predicted anything, across 1999–2025.
* A live paper-trading test and a replay of 2026 Kalshi price history (67 trades) found **no closing line value**: −0.26¢ ± 0.82¢ of line movement toward the model, and −3.9¢ after spread and fees. The model's disagreement with the market doesn't predict where the market moves.
* The project's aim is therefore the best *market-implied* estimate of team skill, not a betting edge.

**Scoreboard** (`scripts/scoreboard.py`). Every model variant is scored, on every daily snapshot, on market prices it wasn't fitted on: held-out current game lines, futures, and forward closing lines. If the file exists, the latest results load below.

In [10]:
sb_path = ROOT / "data" / "scoreboard" / "results.parquet"
if sb_path.exists():
    res = pd.read_parquet(sb_path)
    pooled = lambda g: np.sqrt(np.sum(g.score**2 * g.n) / g.n.sum()) if g.metric.iloc[0] == "RMSE pts" else np.average(g.score, weights=g.n)
    display(res.groupby(["task", "model"]).apply(pooled, include_groups=False).unstack("model").round(3))
else:
    print("no scoreboard results yet: run scripts/scoreboard.py")

model,baseline: today's price,"blind: lines + wins, tau (CMA)",lines,lines + win totals (current),win totals only
task,,,,,
T1 near-term lines,NaN,NaN,3.78,2.17e+00,2.16e+00
T2 futures,NaN,2.10e-02,0.04,2.00e-02,2.00e-02
T2 futures: conference,NaN,3.10e-02,0.06,2.60e-02,2.60e-02
T2 futures: division,NaN,3.00e-03,0.02,5.00e-03,5.00e-03
T2 futures: one_seed,NaN,6.40e-02,0.08,5.70e-02,5.70e-02
T2 futures: super_bowl,NaN,6.00e-02,0.10,5.40e-02,5.30e-02
T3 forward closes,NaN,3.17e+00,2.94,3.00e+00,3.00e+00
T3 forward closes (listed games),2.08,3.17e+00,2.94,3.00e+00,3.00e+00


## 10. Known limitations

* **One rating per team.** When near-term lines and win totals disagree (e.g. a short-term absence), the fit splits the difference. Planned: separate current and rest-of-season ratings.
* **Midpoints, not ranges.** A thin 10¢-wide quote counts like a 1¢ one in the win-total fit. Planned: fit to the bid/ask range.
* **Futures excluded** from the published market rating; the full joint fit didn't converge reliably.
* **Blend calibrated against a lines-only market**, as noted in section 6.
* **Injuries** count only as far as markets price them.
* **2025 Kalshi history** isn't reachable through the public endpoints, so backtests cover 2026 only.

## 11. Changelog

| Date | Change |
|---|---|
| 2026-10-02 | Stage A, market rating (lines + win totals, λ = 300), résumé blend, season sim, Week 4 report. |
| 2026-10-02 | Blend becomes a season-long curve with a luck-neutral résumé term. |
| 2026-10-12 | Game-line paper trading and 2026 CLV backtest: no edge found. |
| 2026-10-14 | **Bug fix:** win-probability lookups had been float32 since 2026-10-02 (`90f736d`). That zeroed the optimizer's gradients, so win totals were silently ignored in the market rating. Fixed with regression tests; Week 6 rankings regenerated. |
| 2026-10-14 | Tied ranks shown as T-*n*; market-vs-results scatter scaled by blend weight; scoreboard added. |